# Session 10: Using Ragas to Evaluate a RAG Application built with LangChain and LangGraph

In the following notebook, we'll be looking at how [Ragas](https://github.com/explodinggradients/ragas) can be helpful in a number of ways when looking to evaluate your RAG applications!

While this example is rooted in LangChain/LangGraph - Ragas is framework agnostic (you don't even need to be using a framework!).

## 🤝 Breakout Room #1
  - Task 1: Installing Required Libraries
  - Task 2: Set Environment Variables
  - Task 3: Synthetic Dataset Generation for Evaluation using Ragas
  - Task 4: Construct our RAG application
  - Task 5: Evaluating our Application with Ragas
  - Task 6: Making Adjustments and Re-Evaluating
  - ***Activity #1: Implement a Different Reranking Strategy***


## Task 1: Installing Required Libraries

If you have not already done so, install the required libraries using the uv package manager:
``` bash

uv sync

```


## Task 2: Set Environment Variables:

We'll also need to provide our API keys.
> NOTE: In addition to OpenAI's models, this notebook will be using Cohere's Reranker - please be sure to [sign-up for an API key!](https://docs.cohere.com/reference/about)

You have two options for supplying your API keys in this session:
- Use environment variables (see Prerequisite #2 in the README.md)
- Provide them via a prompt when the notebook runs

The following code will load all of the environment variables in your `.env`. Then, it checks for the two API keys we need. If they are not there, it will prompt you to provide them.

First, OpenAI's for our LLM/embedding model combination!

Second, Cohere's for our reranking


In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Please enter your Cohere API key!")

## Task 3: Synthetic Dataset Generation for Evaluation using Ragas

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using the Health & Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, and stress management.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [4]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

In [5]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/4 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/9 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

In [6]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How can the Cat-Cow Stretch help with lower ba...,[The Personal Wellness Guide A Comprehensive R...,The Cat-Cow Stretch is recommended for lower b...,single_hop_specifc_query_synthesizer
1,What are partial crunches and how can they hel...,[The Personal Wellness Guide A Comprehensive R...,Partial crunches are an exercise where you lie...,single_hop_specifc_query_synthesizer
2,What is CBT-I and how can it help with insomnia?,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Cognitive Behavioral Therapy for Insomnia (CBT...,single_hop_specifc_query_synthesizer
3,Can magnesium help with sleep issues?,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Magnesium supplements are mentioned as a natur...,single_hop_specifc_query_synthesizer
4,Wut is Chaptr 19 about?,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 19 is about work-life balance and expl...,single_hop_specifc_query_synthesizer
5,Wut is Chapter 13 about?,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 13 is about the science of habit forma...,single_hop_specifc_query_synthesizer
6,What strategies from Chapter 7 and Chapter 19 ...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,Strategies from Chapter 7 to improve sleep qua...,multi_hop_specific_query_synthesizer
7,What natural remedies are recommended in Chapt...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,Chapter 9 recommends several natural remedies ...,multi_hop_specific_query_synthesizer
8,what insomnia in chapter 9 and how work-life b...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,insomnia in chapter 9 is when you have trouble...,multi_hop_specific_query_synthesizer
9,chapter 9 talk bout insomnia and chapter 16 bo...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,chapter 9 say insomnia is trouble sleeping and...,multi_hop_specific_query_synthesizer


## Task 4: Construct our RAG application

Now we'll construct our LangChain RAG, which we will be evaluating using the above created test data!

### R - Retrieval

Let's start with building our retrieval pipeline, which will involve loading the same data we used to create our synthetic test set above.

> NOTE: We need to use the same data - as our test set is specifically designed for this data.

In [7]:
loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

Now that we have our data loaded, let's split it into chunks!

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

447

### ❓ Question #1:

What is the purpose of the `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter`?

##### Answer:

The `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter` sets how many characters at the end of one chunk are repeated or shared at the beginning of next chunk when splitting a text.

Next up, we'll need to provide an embedding model that we can use to construct our vector store.

In [9]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Now we can build our in memory QDrant vector store.

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data_new",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data_new",
    embedding=embeddings,
)

We can now add our documents to our vector store.

In [ ]:
_ = vector_store.add_documents(documents=split_documents)

Let's define our retriever.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Now we can produce a node for retrieval!

In [ ]:
def retrieve(state):
  retrieved_docs = retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

### A - Augmented

Let's create a simple RAG prompt!

In [ ]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### G - Generation

We'll also need an LLM to generate responses - we'll use `gpt-4o-nano` to avoid using the same model as our judge model.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

Then we can create a `generate` node!

In [ ]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

### Building RAG Graph with LangGraph

Let's create some state for our LangGraph RAG graph!

In [ ]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
  question: str
  context: List[Document]
  response: str

Now we can build our simple graph!

> NOTE: We're using `add_sequence` since we will always move from retrieval to generation. This is essentially building a chain in LangGraph.

In [ ]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a test to make sure it's doing what we'd expect.

In [ ]:
response = graph.invoke({"question" : "What exercises help with lower back pain?"})

In [ ]:
response["response"]

'The context provided does not specify any particular exercises that help with lower back pain.'

## Task 5: Evaluating our Application with Ragas

Now we can finally do our evaluation!

We'll start by running the queries we generated usign SDG above through our application to get context and responses.

In [ ]:
for test_row in dataset:
  response = graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [ ]:
dataset.samples[0].eval_sample.response

'The provided context does not include specific instructions on how to perform partial crunches for back pain.'

Then we can convert that table into a `EvaluationDataset` which will make the process of evaluation smoother.

In [ ]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

We'll need to select a judge model - in this case we're using the same model that was used to generate our Synthetic Data.

In [ ]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

Next up - we simply evaluate on our desired metrics!

In [ ]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
baseline_result

Evaluating:   0%|          | 0/54 [00:00<?, ?it/s]

KeyboardInterrupt: 

Exception raised in Job[1]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[26]: AssertionError(LLM must be set)
Exception raised in Job[27]: AssertionError(LLM is not set)
Exception raised in Job[28]: AssertionError(LLM is not initialized)
Exception raised in Job[29]: AssertionError(LLM is not set)
Exception raised in Job[30]: AssertionError(set LLM before use)
Exception raised in Job[31]: AssertionError(LLM is not set)
Exception raised in Job[32]: AssertionError(LLM must be set)
Exception raised in Job[33]: AssertionError(LLM is not set)
Exception raised in Job[34]: AssertionError(LLM is not initialized)
Exception raised in Job[35]: AssertionError(LLM is not set)
Exception raised in Job[36]: As

## Task 6: Making Adjustments and Re-Evaluating

Now that we've got our baseline - let's make a change and see how the model improves or doesn't improve!




We'll first set our retriever to return more documents, which will allow us to take advantage of the reranking.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data_new_chunks",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data_new_chunks",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)

adjusted_example_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

Reranking, or contextual compression, is a technique that uses a reranker to compress the retrieved documents into a smaller set of documents.

This is essentially a slower, more accurate form of semantic similarity that we use on a smaller subset of our documents.

In [ ]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

def retrieve_adjusted(state):
  compressor = CohereRerank(model="rerank-v3.5")
  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=adjusted_example_retriever, search_kwargs={"k": 5}
  )
  retrieved_docs = compression_retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

We can simply rebuild our graph with the new retriever!

In [ ]:
class AdjustedState(TypedDict):
  question: str
  context: List[Document]
  response: str

adjusted_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_adjusted, generate])
adjusted_graph_builder.add_edge(START, "retrieve_adjusted")
adjusted_graph = adjusted_graph_builder.compile()

In [ ]:
response = adjusted_graph.invoke({"question" : "How can I improve my sleep quality?"})
response["response"]

'To improve your sleep quality, consider adopting good sleep hygiene practices such as maintaining a consistent sleep schedule, creating a relaxing bedtime routine, and keeping your bedroom cool, dark, and quiet. Limit screen exposure 1-2 hours before bed and avoid caffeine after 2 PM. Regular exercise can help, but not too close to bedtime. Additionally, following your sleep checklist—like setting room temperature between 65-68°F, using blackout curtains or a sleep mask, and ensuring your mattress and pillows are comfortable—can promote better sleep. You might also explore natural remedies such as relaxation techniques, herbal teas, or meditation, and consider cognitive behavioral therapy for insomnia if needed.'

In [ ]:
import time
import copy

rerank_dataset = copy.deepcopy(dataset)

for test_row in rerank_dataset:
  response = adjusted_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(2) # To try to avoid rate limiting.

In [ ]:
rerank_dataset.samples[0].eval_sample.response

'To perform Shoulder Shrugs to relieve neck and shoulder tension, raise your shoulders toward your ears, hold the position for 5 seconds, then release. Repeat this movement 10 times.'

In [ ]:
rerank_evaluation_dataset = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())

In [ ]:
rerank_result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
rerank_result

Evaluating:   0%|          | 0/54 [00:00<?, ?it/s]

{'context_recall': 0.7407, 'faithfulness': 0.9335, 'factual_correctness': 0.7922, 'answer_relevancy': 0.9704, 'context_entity_recall': 0.3171, 'noise_sensitivity_relevant': 0.0651}

### ❓ Question #2:

Which system performed better, on what metrics, and why?

##### Answer:

Baseline system metrics:
{'context_recall': 0.3148, 'faithfulness': 0.9074, 'factual_correctness': 0.3833, 'answer_relevancy': 0.4273, 'context_entity_recall': 0.4029, 'noise_sensitivity_relevant': 0.2222}

Reranked system metrics:
{'context_recall': 0.7407, 'faithfulness': 0.9335, 'factual_correctness': 0.7922, 'answer_relevancy': 0.9704, 'context_entity_recall': 0.3171, 'noise_sensitivity_relevant': 0.0651}

- The reranked system is better on context_recall, faithfulness, factual_correctness, answer_relevancy and noise_sensitivity_relevant.
- The baseline system is better on context_entity_recall.

The ranked system is better as we increased the chunk_size from 50 --> 500 and chunk_overlap from 0 --> 30 and also increased the top k retrieval from 3 to 20 and then used a reranker to pick top 5 most relevant chunks.
1. Context_recall : The llm did not have the context in the baseline system whereas in the reranked system the llm had the relevant information from the retrieved context, hence the increase in context_recall is justified.
2. Faithfulness and factual_correctness: The reranked system has given a grounded answer based on the source document, so it hallucinates less and answers more accurately. The baseline system`s faithfulness is also high as it has followed the user prompt and not used external knowledge to answer, based on the context provided, it has produced a conservative reply.
3. answer_relevancy: The reranked system's answer addresses the question asked, hence increase in this metric.
4. noise_sensitivity_relevant: Reranker ranks the top 20 retrieved chunks and filters to only 5, noisy chunks are likely filtered out.

### ❓ Question #3:

What are the benefits and limitations of using synthetic data generation for RAG evaluation? Consider both the practical advantages and potential pitfalls.

##### Answer:

Benefits:
1. We can create many Q&A sets without needing human to write them.
2. Cheaper and faster once the test generator is setup.
3. You dont have to use real user data, can mock it, hence helps in provacy.
4. We can test many variations/use cases with same dataset.
5. Can be tested on various metrics.

Limitations:
1. Results might not be 100% accurate as we are not using real production data.
2. There can be a bias in evaluation results when we use same models to generate test data and to evaluate.
3. Hallucination is always possible with LLMs, hence the metriccs can be against wrong references in such cases.

### ❓ Question #4:

If you were building a production wellness assistant, which Ragas metrics would be most important to optimize for and why? Consider the healthcare/wellness domain specifically.

##### Answer:

For a healthcare & wellness domain, I think below Ragas metrics are important and needs to be carefully designed:
1. Faithfulness: Answers should always be grounded in the retrieved context, if this goes wrong then it increases the risk of misleading or dangerous health guidance.
2. Context recall & context precision: Missing or diluted context can lead to wrong advices. Hence it is important to retrieve the right information from the knowledge base.
3. Answer relevancy:  It is important that the assistant stays on topic and actionable. Generic or off topic answers reduce the user's trust.

Even with the best design and good metrics, for health and wellness, it is always best to add a disclaimer.

## Activity #1: Implement a Different Reranking Strategy

In this activity, you'll experiment with different reranking parameters or strategies to see how they affect the evaluation metrics.

**Requirements:**
1. Modify the `retrieve_adjusted` function to use different parameters (e.g., change `k` values, try different top_n for reranking)
2. Or implement a different retrieval enhancement strategy (e.g., hybrid search, query expansion)
3. Run the evaluation and compare results with the baseline and reranking results above
4. Document your findings in the markdown cell below

In [ ]:
### YOUR CODE HERE ###

# Implement your custom retrieval strategy here
# Example: modify retrieve_adjusted with different parameters

import time
import copy
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

custom_adjusted_retriever = vector_store.as_retriever(search_kwargs={"k": 15})

def retrieve_custom(state):
  compressor = CohereRerank(model="rerank-v3.5")
  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=custom_adjusted_retriever, search_kwargs={"k": 3}
  )
  retrieved_docs = compression_retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

class CustomAdjustedState(TypedDict):
  question: str
  context: List[Document]
  response: str

custom_adjusted_graph_builder = StateGraph(CustomAdjustedState).add_sequence([retrieve_custom, generate])
custom_adjusted_graph_builder.add_edge(START, "retrieve_custom")
custom_adjusted_graph = custom_adjusted_graph_builder.compile()

response = custom_adjusted_graph.invoke({"question" : "How can I improve my sleep quality?"})
response["response"]

rerank_dataset = copy.deepcopy(dataset)

for test_row in rerank_dataset:
  response = custom_adjusted_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(2) # To try to avoid rate limiting.


custom_rerank_evaluation_dataset = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())

rerank_result = evaluate(
    dataset=custom_rerank_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
rerank_result


Evaluating:   0%|          | 0/54 [00:00<?, ?it/s]

### Activity #1 Findings:

*Document your findings here: What strategy did you try? How did it compare to the baseline and reranking results?*